<a href="https://colab.research.google.com/github/faiqakashif82-netizen/FlyRank-MachineLearning-Internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/faiqakashif82-netizen/FlyRank-MachineLearning-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
%pip install -q duckdb huggingface_hub pandas scikit-learn matplotlib

In [ ]:
import duckdb
import pandas as pd
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"

# Correct paths, based on the real file listing
print("dim_clients:", con.sql(f"SELECT COUNT(*) FROM read_parquet('{BASE}/dim_clients.parquet')").fetchone()[0])
print("dim_content:", con.sql(f"SELECT COUNT(*) FROM read_parquet('{BASE}/dim_content.parquet')").fetchone()[0])
print("fact_content_daily_performance (all months):", con.sql(f"SELECT COUNT(*) FROM read_parquet('{BASE}/fact_content_daily_performance/**/*.parquet')").fetchone()[0])
print("fact_content_query_90d:", con.sql(f"SELECT COUNT(*) FROM read_parquet('{BASE}/fact_content_query_90d.parquet')").fetchone()[0])

dim_clients: 104
dim_content: 519606


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_content_daily_performance (all months): 78835655
fact_content_query_90d: 2414248


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Unit of analysis

One row = one pseudonymized content item (`content_hash_id`) for one pseudonymized client (`client_hash_id`) on one calendar day (`report_date`).

The row contains that day's organic search performance, including GSC impressions, clicks, and average position.

### Time window

I develop and verify this contract using `month=2026-03`.

I use March 2026 as the mid-panel development month. I do not use the `_sample` table for label development because it represents the final month (June 2026), which should be treated as a sealed test month.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Features

These are the five features I plan to use:

1. `avg_clicks_28d` — average GSC clicks from the previous 28 days.
2. `avg_impressions_28d` — average GSC impressions from the previous 28 days.
3. `avg_position_28d` — average GSC average position from the previous 28 days.
4. `ctr_28d` — clicks divided by impressions over the previous 28 days.
5. `content_age_days` — number of days since the content was created.

These features are intended to use information that was available before the prediction/decision moment.

### Label / proxy

`is_declining_label` is a simple proxy label.

It is `True` when the content's average GSC impressions in March 2026 are lower than its average GSC impressions in February 2026.

This is a current-window proxy rather than a final future-window production label.

### Context

- `client_hash_id` — pseudonymous client identifier used for grouping and joins.
- `content_hash_id` — pseudonymous content identifier used for grouping and joins.

These identifiers are context fields, not model features.

### Excluded

I exclude `sessions_ai` and the other `ai_*` columns.

The lane guidance says AI-referral sessions are very sparse in this warehouse, so I am excluding them from this first search-intelligence feature set rather than treating sparse AI-referral activity as a main signal.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Every claim above gets a query cell here, run on month=2026-03.

### Query 1 — Grain

I check whether the combination of `client_hash_id`, `content_hash_id`, and `report_date` appears more than once.

If the duplicate count is zero, the observed data supports the contract that one row represents one content item for one client on one calendar day.

In [ ]:
grain_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT (client_hash_id, content_hash_id, report_date)) AS distinct_grain_rows
    FROM read_parquet(
        '{BASE}/fact_content_daily_performance/month=2026-03/data_0.parquet'
    )
""").df()

grain_check


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,distinct_grain_rows
0,9841378,9841378


### Query 2 — March slice size and date span

I measure the size of the March 2026 slice and its observed date range.

This verifies how many rows are available in the development month and which dates are actually present.

In [ ]:
slice_stats = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT content_hash_id) AS unique_content_items,
        COUNT(DISTINCT client_hash_id) AS unique_clients,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM read_parquet(
        '{BASE}/fact_content_daily_performance/month=2026-03/data_0.parquet'
    )
""").df()

slice_stats

,row_count,unique_content_items,unique_clients,first_date,last_date
0,9841378,331437,55,2026-03-01,2026-03-31


### Query 3 — Availability

I check how many March rows have GA4 data available.

The availability check uses `IS TRUE` explicitly, so only rows where the availability flag is actually True are counted as available.

In [ ]:
availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
        ) AS ga4_available_rows,
        COUNT(*) FILTER (
            WHERE ga4_data_available IS NOT TRUE
        ) AS ga4_unavailable_or_null_rows
    FROM read_parquet(
        '{BASE}/fact_content_daily_performance/month=2026-03/data_0.parquet'
    )
""").df()

availability_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,ga4_available_rows,ga4_unavailable_or_null_rows
0,9841378,413966,9427412


### Five-feature frame

I build five features using information available before the March decision window.

- `avg_clicks_28d` — available at the decision moment because it uses historical GSC clicks.
- `avg_impressions_28d` — available at the decision moment because it uses historical GSC impressions.
- `avg_position_28d` — available at the decision moment because it uses historical GSC average position.
- `ctr_28d` — available at the decision moment because it is calculated from historical clicks and impressions.
- `content_age_days` — available at the decision moment because the content creation date is already known.

In [ ]:
features_month = con.sql(f"""
    WITH history AS (
        SELECT
            f.client_hash_id,
            f.content_hash_id,
            AVG(f.gsc_clicks) AS avg_clicks_28d,
            AVG(f.gsc_impressions) AS avg_impressions_28d,
            AVG(f.gsc_avg_position) AS avg_position_28d,
            SUM(f.gsc_clicks) / NULLIF(SUM(f.gsc_impressions), 0) AS ctr_28d
        FROM read_parquet(
            '{BASE}/fact_content_daily_performance/month=2026-02/data_0.parquet'
        ) f
        GROUP BY
            f.client_hash_id,
            f.content_hash_id
    )

    SELECT
        h.client_hash_id,
        h.content_hash_id,
        h.avg_clicks_28d,
        h.avg_impressions_28d,
        h.avg_position_28d,
        h.ctr_28d,
        DATE_DIFF(
            'day',
            d.content_created_date,
            DATE '2026-03-01'
        ) AS content_age_days
    FROM history h
    JOIN read_parquet(
        '{BASE}/dim_content.parquet'
    ) d
        ON h.content_hash_id = d.content_hash_id
""").df()

print(features_month.shape)
features_month.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(321546, 7)


,client_hash_id,content_hash_id,avg_clicks_28d,avg_impressions_28d,avg_position_28d,ctr_28d,content_age_days
0,client_3ffa76342f366962,content_fb84747a57b8b665,0.0,0.0,NaN,NaN,176
1,client_3ffa76342f366962,content_feccf822ac21326e,0.0,0.0,NaN,NaN,176
2,client_3ffa76342f366962,content_17cf93c10413ebe9,0.0,0.0,NaN,NaN,176
3,client_3ffa76342f366962,content_a9905735266f8697,0.0,0.0,NaN,NaN,176
4,client_3ffa76342f366962,content_31c34765e7bba2f0,0.0,0.0,NaN,NaN,176


### March decline proxy label

I define a simple proxy label for the March development window.

`is_declining_label` is True when a content item's average impressions in March are lower than its average impressions in February.

This label is used for the leakage demonstration in this notebook.

In [ ]:
labeled = con.sql(f"""
    WITH curr AS (
        SELECT
            client_hash_id,
            content_hash_id,
            AVG(gsc_impressions) AS impressions_curr
        FROM read_parquet(
            '{BASE}/fact_content_daily_performance/month=2026-03/data_0.parquet'
        )
        GROUP BY client_hash_id, content_hash_id
    ),
    prev AS (
        SELECT
            client_hash_id,
            content_hash_id,
            AVG(gsc_impressions) AS impressions_prev
        FROM read_parquet(
            '{BASE}/fact_content_daily_performance/month=2026-02/data_0.parquet'
        )
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT
        c.client_hash_id,
        c.content_hash_id,
        c.impressions_curr,
        p.impressions_prev,
        (c.impressions_curr < p.impressions_prev) AS is_declining_label
    FROM curr c
    JOIN prev p
        ON c.client_hash_id = p.client_hash_id
        AND c.content_hash_id = p.content_hash_id
""").df()

labeled.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,impressions_curr,impressions_prev,is_declining_label
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,210.419355,152.500000,False
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,14.612903,15.714286,True
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,181.612903,188.250000,True
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,159.483871,238.928571,True
4,client_73cda7b4e4f265ea,content_f39be42b42a4e8f6,1.354839,0.857143,False


### Feature and label frame

I join the historical feature frame with the March proxy label using the pseudonymous client and content IDs.

In [ ]:
frame = features_month.merge(
    labeled,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

print(frame.shape)
frame.head()

(303572, 10)


,client_hash_id,content_hash_id,avg_clicks_28d,avg_impressions_28d,avg_position_28d,ctr_28d,content_age_days,impressions_curr,impressions_prev,is_declining_label
0,client_3ffa76342f366962,content_fb84747a57b8b665,0.0,0.0,NaN,NaN,176,0.0,0.0,False
1,client_3ffa76342f366962,content_feccf822ac21326e,0.0,0.0,NaN,NaN,176,0.0,0.0,False
2,client_3ffa76342f366962,content_17cf93c10413ebe9,0.0,0.0,NaN,NaN,176,0.0,0.0,False
3,client_3ffa76342f366962,content_a9905735266f8697,0.0,0.0,NaN,NaN,176,0.0,0.0,False
4,client_3ffa76342f366962,content_31c34765e7bba2f0,0.0,0.0,NaN,NaN,176,0.0,0.0,False


### Deliberate leakage experiment

I now add one intentionally leaked feature derived directly from the label.

This is not an acceptable production feature. I am adding it only to demonstrate the leakage problem.

Because the leaked feature contains the answer, the quick model score should become artificially close to perfect.

I then remove the leaked feature and calculate the honest score using only the five historical features.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

leak_frame = frame.copy()

# Deliberately create a leaked feature from the label
leak_frame["leaked_is_declining"] = leak_frame["is_declining_label"].astype(int)

X = leak_frame[["leaked_is_declining"]]
y = leak_frame["is_declining_label"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

leak_model = DecisionTreeClassifier(
    random_state=42,
    max_depth=2
)

leak_model.fit(X_train, y_train)

leaked_score = accuracy_score(
    y_test,
    leak_model.predict(X_test)
)

print("Score with leaked feature:", leaked_score)

Score with leaked feature: 1.0


### Remove the leaked feature

The score of 1.0 is artificially high because `leaked_is_declining` directly contains the label.

I now remove the leaked feature and calculate the score using only the five planned features.

In [ ]:
five_features = [
    "avg_clicks_28d",
    "avg_impressions_28d",
    "avg_position_28d",
    "ctr_28d",
    "content_age_days"
]

clean_frame = frame.dropna(
    subset=five_features + ["is_declining_label"]
).copy()

X = clean_frame[five_features]
y = clean_frame["is_declining_label"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

honest_model = DecisionTreeClassifier(
    random_state=42,
    max_depth=3
)

honest_model.fit(X_train, y_train)

honest_predictions = honest_model.predict(X_test)

honest_score = accuracy_score(
    y_test,
    honest_predictions
)

print("Score after removing leaked feature:", honest_score)

Score after removing leaked feature: 0.5876583149779736


### Leakage lesson

The leaked feature produced a perfect score of 1.0 because it directly contained the label.

After removing the leaked feature, the score dropped to 0.588. This shows that the perfect score was caused by label leakage rather than genuine predictive signal.

The leaked feature is therefore removed and is not kept as a production feature. Only information that would be available at the decision moment should be used as a feature.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Limitation

A key limitation of this slice is that historical coverage is not equally complete for every content item. Some content items have less usable history than others, so the historical features may represent different amounts of information across items.

The March 2026 slice is also only one month, so the results should be treated as directional decision-support rather than a complete measure of long-term search performance.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.